In [1]:
import sys
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import RandomOverSampler

sys.path.append(str(Path.cwd().parent / "src"))
from credit_risk import build_model, evaluate, split

---

## Split the Data into Training and Testing Sets

### Step 1: Read the `lending_data.csv` data from the `Resources` folder into a Pandas DataFrame.

In [2]:
lending_data_df = pd.read_csv("Resources/lending_data.csv")
lending_data_df.head()

,loan_size,interest_rate,borrower_income,debt_to_income,num_of_accounts,derogatory_marks,total_debt,loan_status
0,10700.0,7.672,52800,0.431818,5,1,22800,0
1,8400.0,6.692,43600,0.311927,3,0,13600,0
2,9000.0,6.963,46100,0.349241,3,0,16100,0
3,10700.0,7.664,52700,0.430740,5,1,22700,0
4,10800.0,7.698,53000,0.433962,5,1,23000,0


### Step 2: Create the labels set (`y`)  from the “loan_status” column, and then create the features (`X`) DataFrame from the remaining columns.

In [3]:
y = lending_data_df["loan_status"]
X = lending_data_df.drop(columns="loan_status")

In [4]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: loan_status, dtype: int64

In [5]:
X.head()

,loan_size,interest_rate,borrower_income,debt_to_income,num_of_accounts,derogatory_marks,total_debt
0,10700.0,7.672,52800,0.431818,5,1,22800
1,8400.0,6.692,43600,0.311927,3,0,13600
2,9000.0,6.963,46100,0.349241,3,0,16100
3,10700.0,7.664,52700,0.430740,5,1,22700
4,10800.0,7.698,53000,0.433962,5,1,23000


### Step 3: Check the balance of the labels variable (`y`) by using the `value_counts` function.

In [6]:
y.value_counts()

loan_status
0    75036
1     2500
Name: count, dtype: int64

### Step 4: Split the data into training and testing datasets by using `train_test_split`.

The split is stratified on `loan_status`, so the high-risk share (3.2%) is the same in both sets.

In [7]:
X_train, X_test, y_train, y_test = split(X, y)
pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()})

,train,test
loan_status,,
0,56277,18759
1,1875,625


---

## Create a Logistic Regression Model with the Original Data

###  Step 1: Fit a logistic regression model by using the training data (`X_train` and `y_train`).

Features are standardized first; without scaling, the `lbfgs` solver does not converge on this data.

In [8]:
baseline = build_model().fit(X_train, y_train)
baseline

,steps,"[('scale', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False


### Step 2: Save the predictions on the testing data labels by using the testing feature data (`X_test`) and the fitted model.

In [9]:
baseline_results = evaluate(baseline, X_test, y_test)

### Step 3: Evaluate the model’s performance by doing the following:

* Calculate the accuracy score of the model.

* Generate a confusion matrix.

* Print the classification report.

In [10]:
print(f"Balanced accuracy: {baseline_results['balanced_accuracy']:.3f}")

Balanced accuracy: 0.986


In [11]:
print(baseline_results["confusion_matrix"])

[[18669    90]
 [   14   611]]


In [12]:
print(baseline_results["report"])

              precision    recall  f1-score   support

     healthy      0.999     0.995     0.997     18759
   high-risk      0.872     0.978     0.922       625

    accuracy                          0.995     19384
   macro avg      0.935     0.986     0.959     19384
weighted avg      0.995     0.995     0.995     19384



### Step 4: Answer the following question.

**Question:** How well does the logistic regression model predict both the `0` (healthy loan) and `1` (high-risk loan) labels?

**Answer:** It does well for our healthy loans, but when it comes to our unhealthy loans, our predictions become less accurate. This is because of the difference in available data between our healthy and unhealthy loans: we have 18,759 data points for healthy loans in the test set but only 625 unhealthy loans. But as you can see, our model does well in both categories in our weighted average.

---

## Predict a Logistic Regression Model with Resampled Training Data

### Step 1: Use the `RandomOverSampler` module from the imbalanced-learn library to resample the training data to a 0.18 high-risk to healthy ratio.

In [13]:
X_over, y_over = RandomOverSampler(sampling_strategy=0.18, random_state=1).fit_resample(X_train, y_train)

In [14]:
y_over.value_counts()

loan_status
0    56277
1    10129
Name: count, dtype: int64

### Step 2: Use the `LogisticRegression` classifier and the resampled data to fit the model and make predictions.

In [15]:
# The pipeline applies the same oversampling internally, to the training data only
oversampled = build_model(oversample_ratio=0.18).fit(X_train, y_train)
oversampled_results = evaluate(oversampled, X_test, y_test)

### Step 3: Evaluate the model’s performance by doing the following:

* Calculate the accuracy score of the model.

* Generate a confusion matrix.

* Print the classification report.

In [16]:
print(f"Balanced accuracy: {oversampled_results['balanced_accuracy']:.3f}")

Balanced accuracy: 0.994


In [17]:
print(oversampled_results["confusion_matrix"])

[[18668    91]
 [    5   620]]


In [18]:
print(oversampled_results["report"])

              precision    recall  f1-score   support

     healthy      1.000     0.995     0.997     18759
   high-risk      0.872     0.992     0.928       625

    accuracy                          0.995     19384
   macro avg      0.936     0.994     0.963     19384
weighted avg      0.996     0.995     0.995     19384



### Step 4: Answer the following question

**Question:** How well does the logistic regression model, fit with oversampled data, predict both the `0` (healthy loan) and `1` (high-risk loan) labels?

**Answer:** Our oversampled model performs very similarly to the model trained on the original data. However, the difference here is that it does better on the unhealthy recall score, and yet our precision remains in the same range. Even though we oversample a lot more training data to train on, its ability to avoid false positives remains unchanged, as the number does not change, indicating we need more unhealthy data points for our training model, since random oversampling helps with the recall but does so by repeatedly copying the same unhealthy data points.